# 00 — Setup, confirmed-contract re-verification, memory gate, smoke test

**Project:** RLVR plasticity, exp2 Colab variant (Math -> Simulation, Qwen2.5-7B, LoRA, **merged with the WIN4070 track's group-8 fix** — see `EXPERIMENT_2_COLAB_PLAN.md` §0/§1.4) · **Owner:** Aaron (Person 4)

Unlike an earlier draft of this notebook, this does NOT discover the GURU schema from scratch — the WIN4070 track already did that for real (`data/guru_schema_audit.json`) and this notebook's data/reward code (`src/guru_data.py`, `src/guru_reward.py`) is built directly against that confirmed contract. What follows re-verifies it (spot-checks loaded rows, re-runs the token audit under this model's own tokenizer) rather than re-discovering it. Nothing here trains beyond a 2-update smoke test. **Do not run 01 until every gate below has passed and this notebook's outputs are committed.**

In [ ]:
import subprocess, sys, os, json
from pathlib import Path

# Private repo: reads a token from Colab's own Secrets store (key icon,
# left sidebar) — add one named GITHUB_TOKEN (a GitHub PAT with repo read
# access) before running this cell. The token is never written to this
# notebook's source and this cell never prints it.
from google.colab import userdata
try:
    _token = userdata.get('GITHUB_TOKEN')
except Exception:
    _token = None
if not _token:
    raise RuntimeError(
        'Add a GITHUB_TOKEN secret (key icon, left sidebar) with repo read '
        'access to WYR186/RLVR, enable notebook access for it, then re-run.')
REPO_URL = f'https://{_token}@github.com/WYR186/RLVR.git'
REPO_DIR = '/content/RLVR'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
# strip the token back out of the stored remote URL immediately — no need
# to leave it sitting in .git/config for the rest of the session
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin',
                'https://github.com/WYR186/RLVR.git'], check=True)
del _token, REPO_URL  # don't leave the token bound in the notebook's live namespace

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
MODEL_ID, MODEL_REVISION = CONFIG['model_id'], CONFIG['model_revision']
DATASET_REVISION = CONFIG['dataset']['revision']
print('config loaded:', CONFIG['experiment'])
print('merge note:', CONFIG['merge_note'])

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

## Step 1-3 — load the confirmed contract, spot-check rows

`load_all_records` renders every prompt through THIS model's chat template and computes token counts with THIS model's tokenizer — the field names and file paths are pinned/confirmed, but the counts below are still real numbers for this model, not copy-pasted from the 0.5B track.

In [ ]:
math_rows, sim_rows = guru_data.load_all_records(
    MODEL_ID, MODEL_REVISION, DATASET_REVISION,
    stage_a_prompt_suffix=None)
print('Math (stage A) rows:', len(math_rows))
print('Simulation/CodeIO (stage B) rows:', len(sim_rows))
print()
print('--- sample Math row ---')
sample = math_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})
print()
print('--- sample Simulation row ---')
sample = sim_rows[0]
print({k: (v[:300] if isinstance(v, str) else v) for k, v in sample.items() if k != 'extra_info'})

**Sanity check before continuing:** do the two counts above look like the confirmed audit's `stage_a_count`/`stage_b_count` (`data/guru_schema_audit.json`), and does each sample row have a real rendered prompt (not an empty string or a raw message-list repr) and a real ground_truth? If not, stop and investigate — do not proceed on a loader that silently produced garbage.

## Step 4 — token-length audit under this model's tokenizer (GATE 0a re-verification)

In [ ]:
audit_a = guru_data.token_stats(math_rows)
audit_b = guru_data.token_stats(sim_rows)
print('stage A (Math):', audit_a)
print('stage B (Simulation):', audit_b)

gate_0a_threshold = CONFIG['gates']['phase0a_stage_b_p95_prompt_tokens_max']
if audit_b['p95'] > gate_0a_threshold:
    raise SystemExit(
        f"GATE 0a STOP: stage-B p95={audit_b['p95']} > {gate_0a_threshold}. "
        'Escalate GPU tier — do not shrink the batch to force a fit.')
print('GATE 0a: PASS (confirmed audit already found stage-B p95 well under 1024; this just re-verifies)')

## Step 5 — freeze splits (train/eval/probe), model- and geometry-specific

In [ ]:
splits = guru_data.build_exp2_splits(
    MODEL_ID, MODEL_REVISION,
    stage_a_token_limit=CONFIG['stage_a']['token_filter_max'],
    stage_b_token_limit=CONFIG['stage_b']['token_filter_max'],
    stage_b_eval_questions=CONFIG['stage_b']['eval_questions'],
    n_probe=CONFIG['measurement']['probe_questions'],
    dataset_revision=DATASET_REVISION, seed=CONFIG['seed'],
    out_name='exp2_colab_splits.json')
print('stage_a_train:', len(splits['stage_a_train_ids']),
      '| stage_b_train:', len(splits['stage_b_train_ids']),
      '| stage_b_eval:', len(splits['stage_b_eval_ids']),
      '| probe:', splits['probe_actual'], '/', splits['probe_requested'])
if 'probe_shortfall_note' in splits:
    print('WARNING:', splits['probe_shortfall_note'])

## Gate C0 — GPU memory calibration at the REAL group-8 geometry

This is the first time group 8 will run to completion anywhere in this project (the WIN4070 track's own group-8 attempt OOM'd before finishing its smoke). Escalate tier if it doesn't fit — do not shrink `num_generations`/batch below the config to force an L4 fit.

In [ ]:
sa = CONFIG['stage_a']
smoke_ds = guru_data.to_hf_dataset(math_rows[:8])

gate_c0 = pipeline.gate_c0_memory_probe(
    MODEL_ID, CONFIG['peft'], smoke_ds, sa['reward_mode'],
    num_generations=sa['num_generations'],
    per_device_batch=sa['per_device_train_batch_size'],
    grad_accum=sa['gradient_accumulation_steps'],
    max_completion_length=sa['max_completion_length'],
    device='cuda', min_headroom_pct=CONFIG['gates']['gate_c0_memory_headroom_min_pct'],
    learning_rate=sa['learning_rate'])
print(gate_c0)
if not gate_c0['gate_pass']:
    print('Gate C0: escalate to A100 (switch the Colab runtime, then re-run this cell). '
          'This was the EXPECTED outcome per the plan (§1, GPU tier row) - not a surprise.')
else:
    print('Gate C0: PASS on current tier.')

## Phase 0 step 7-8 — smoke test + tightened sparse-reward preflight (GATE 0b)

16 frozen Stage-A prompts x 8 generations, 8 frozen Stage-B prompts x 8 generations. STOP unless >=2 groups have variable COMBINED reward on EACH stage; exact-channel variance is tracked and reported separately (`FINDING_GROUP_SIZE_REWARD_VARIANCE.md` — combined variance alone overstates how much of the group-8 gain is real reasoning signal vs. format-shaping noise).

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')

stage_a_preflight_rows = [
    r for r in math_rows if r['id'] in set(splits['stage_a_train_ids'])
][:CONFIG['gates']['phase0b_stage_a_preflight_prompts']]
stage_b_preflight_rows = [
    r for r in sim_rows if r['id'] in set(splits['stage_b_train_ids'])
][:CONFIG['gates']['phase0b_stage_b_preflight_prompts']]

preflight_a = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_a_preflight_rows, sa['reward_mode'],
    num_generations=sa['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])
sb = CONFIG['stage_b']
preflight_b = pipeline.guru_sparse_reward_preflight(
    model, tokenizer, stage_b_preflight_rows, sb['reward_mode'],
    num_generations=sb['num_generations'],
    min_variable_groups=CONFIG['gates']['phase0b_min_variable_groups'])

for name, pf in [('Stage A (Math)', preflight_a), ('Stage B (Simulation)', preflight_b)]:
    print(f"{name}: combined-variable groups {pf['groups_with_combined_variance']}/{pf['n_prompts']}, "
          f"exact-variable groups {pf['groups_with_exact_variance']}/{pf['n_prompts']}, "
          f"has_grpo_signal={pf['has_grpo_signal']}")

if not (preflight_a['has_grpo_signal'] and preflight_b['has_grpo_signal']):
    raise SystemExit(
        'GATE 0b STOP: fewer than the required variable groups on at least one stage. '
        'Preserve this preflight result and ask the team. Do not add extra shaping reward '
        'beyond the registered exact_plus_boxed_format_0.1 mode; if the failure looks like '
        'a format-compliance problem specifically, consider the Instruct fallback (config '
        "'model_variant_contingency') and log the deviation - don't silently switch.")
print('GATE 0b: PASS on both stages')

**Format-following check (Base vs Instruct contingency, plan §1/§8 item 4):** scan `preflight_a['groups'][*]['completion_tails']` above for repeated failure to emit a well-formed `\boxed{}`. If most completions never attempt the format, that is the specific signal the WIN4070 track's switch to Instruct was responding to at 0.5B scale — flag it before spending Phase 1 compute on a base model that can't be scored.

In [ ]:
model, tokenizer = pipeline.build_peft_model(MODEL_ID, CONFIG['peft'], device='cuda')
smoke_a = guru_data.to_hf_dataset(stage_a_preflight_rows[:8])
smoke_b = guru_data.to_hf_dataset(stage_b_preflight_rows[:8])

for label, ds, mode, geom in [
    ('stage A smoke', smoke_a, sa['reward_mode'], sa),
    ('stage B smoke', smoke_b, sb['reward_mode'], sb),
]:
    from trl import GRPOConfig, GRPOTrainer
    cfg = GRPOConfig(
        output_dir=f'/tmp/exp2_smoke_{label.replace(" ", "_")}', seed=42, max_steps=2,
        learning_rate=geom['learning_rate'], per_device_train_batch_size=geom['per_device_train_batch_size'],
        gradient_accumulation_steps=geom['gradient_accumulation_steps'], num_generations=geom['num_generations'],
        beta=geom['beta'], max_completion_length=geom['max_completion_length'],
        bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=1, save_strategy='no', report_to='none')
    trainer = GRPOTrainer(model=model, args=cfg, train_dataset=ds,
                          reward_funcs=guru_reward.select_reward_fn(mode), processing_class=tokenizer)
    trainer.train()
    print(label, 'completed 2/2 smoke updates OK')

## Commit reminder

Commit `data/exp2_colab_splits.json` with message prefix `exp2-colab:`. Log this phase's wall time, GPU tier, and Colab compute-unit cost in `eaaj-pilot/compute_log.md` before moving to notebook 01.